# Week 3: Prompts as Engineering Artifacts - Dalien Cable

Support ticket triage for a connected diabetes management app (CGM sensor + smart pen cap that detects removal/replacement from an insulin pen to log a treatment). Six categories: device, treatment, lifestyle, supply, account, caregiver. Each ticket also gets a severity level (P1-P4).

`diabetes_prompt.md` (v1) scores severity by reading the severity table literally. `diabetes_prompt_expanded.md` (v2) adds one instruction: weigh the worst plausible downstream consequence to the patient vs just focusing on the stated problem. That is the main edit that is being tested.

Runs without an API key: model calls fall back to labeled fixtures when `GEMINI_API_KEY` is unset. Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [1]:
import os, json, pathlib
def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from google import genai
    client = genai.Client(api_key=key)
    response = client.models.generate_content(model=model, contents=messages[0]['content'])
    return response.text

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [2]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Structured prompts, versioned as files

Both prompts share the same six categories and the same seven few-shot examples' worth of reasoning style. v1 scores severity against a plain table. v2 adds the downstream-consequence instruction plus one extra demonstrating example (a silently-dropped dose log, scored P1). That is the isolated edit between versions.

In [3]:
PROMPT_V1 = pathlib.Path('prompts/diabetes_prompt.md').read_text()
PROMPT_V2 = pathlib.Path('prompts/diabetes_prompt_expanded.md').read_text()

tests = [
  {'id': 1, 'ticket': 'I just received an Rx change from my endocrinologist to switch my long-acting pen from Lantus Solostar to Basaglar KwikPen. How do I edit the pen type and dose in the app?', 'expected_category': 'treatment', 'expected_severity_v1': 'P3', 'expected_severity_v2': 'P3', 'why': 'pen switch request, administrative not malfunction'},
  {'id': 2, 'ticket': "my sensor keeps losing connection with the app at night causing the critical alert to go off. It happens many times a night. I'm exhausted and have started turning off the phone completely so I can sleep.", 'expected_category': 'device', 'expected_severity_v1': 'P2', 'expected_severity_v2': 'P1', 'why': 'disabled critical alert due to repeated disconnection'},
  {'id': 3, 'ticket': "My CGM transmitter completely died and won't restart. I have zero visibility into my glucose readings right now.", 'expected_category': 'device', 'expected_severity_v1': 'P1', 'expected_severity_v2': 'P1', 'why': 'total loss of glucose monitoring'},
  {'id': 4, 'ticket': 'My provider adjusted my basal rate to be higher overnight starting this week, how do I update that in my settings?', 'expected_category': 'treatment', 'expected_severity_v1': 'P3', 'expected_severity_v2': 'P3', 'why': 'routine regimen change update'},
  {'id': 5, 'ticket': "What's a good post-workout snack to prevent a low blood sugar?", 'expected_category': 'lifestyle', 'expected_severity_v1': 'P4', 'expected_severity_v2': 'P4', 'why': 'general lifestyle habit question'},
  {'id': 6, 'ticket': "I'm switching from an iPhone to an Android phone, can I copy my account and settings across?", 'expected_category': 'account', 'expected_severity_v1': 'P4', 'expected_severity_v2': 'P4', 'why': 'account transfer request'},
  {'id': 7, 'ticket': "I dropped my pen cap and it cracked. It still seems to be working but I'd like a replacement sent ASAP.", 'expected_category': 'supply', 'expected_severity_v1': 'P3', 'expected_severity_v2': 'P3', 'why': 'precautionary replacement request'},
  {'id': 8, 'ticket': 'How do I remove my ex-spouse as an authorized caregiver on my account?', 'expected_category': 'caregiver', 'expected_severity_v1': 'P3', 'expected_severity_v2': 'P3', 'why': 'caregiver access management request'},
  {'id': 9, 'ticket': "I was locked out of my account after too many failed login attempts and can't see my glucose data right now.", 'expected_category': 'account', 'expected_severity_v1': 'P4', 'expected_severity_v2': 'P2', 'why': 'locked out of viewing glucose data'},
  {'id': 10, 'ticket': 'My step count in the app seems inaccurate compared to my fitness tracker.', 'expected_category': 'device', 'expected_severity_v1': 'P4', 'expected_severity_v2': 'P4', 'why': 'cosmetic tracking discrepancy'}
]
print('prompt versions: 2 | test cases:', len(tests))

prompt versions: 2 | test cases: 10


## Part 2: Test suite and metrics

Ten input/expected-output pairs. Category is checked with exact-match. Severity is checked with exact-match against a version-specific expected value, since v1 and v2 are predicted to diverge on two cases (#2 and #9). Rationale quality is checked with semantic similarity against a short expected-reasoning phrase.

In [4]:
# Labeled fixtures stand in for model output when LIVE is False.
FIX_V1 = {1: ('treatment', 'This is about updating the treatment plan to match a new prescription, not a malfunction.', 'P3', "The app's records are temporarily out of sync with the actual prescription, but the patient still has clear guidance from their endocrinologist."), 2: ('device', 'The sensor is failing to maintain a connection to the app, which is a hardware or connectivity malfunction.', 'P2', 'Repeated disconnections create a real-time monitoring gap that requires a manual workaround.'), 3: ('device', 'The transmitter has stopped functioning entirely, which is a hardware failure.', 'P1', 'The patient has zero visibility into their glucose levels, which is a complete loss of monitoring.'), 4: ('treatment', 'This is a request to update the app to reflect a new dosing regimen from the provider.', 'P3', 'This is a configuration update tied to a real regimen change, not a malfunction, and the patient still has clear guidance from their provider.'), 5: ('lifestyle', 'This is a general food and activity question, not tied to a specific dose.', 'P4', 'This has no bearing on glucose tracking or insulin delivery.'), 6: ('account', 'This is a request to transfer profile and settings across devices, an administrative task.', 'P4', 'This is a routine account transfer request with no impact on glucose tracking or insulin delivery.'), 7: ('supply', 'This is about getting a replacement for a damaged accessory.', 'P3', "The current pen cap is still functioning, so there's a buffer before this becomes urgent."), 8: ('caregiver', "This is about managing who has authorized access to the account, not the account's own billing or profile.", 'P3', "Removing unwanted access is important, but the patient's own monitoring and alerts are unaffected in the meantime."), 9: ('account', 'This is a login and access issue, not a device malfunction.', 'P4', 'This is a routine account access issue.'), 10: ('device', 'This is a data accuracy bug in a tracking feature within the app.', 'P4', 'This is a cosmetic discrepancy unrelated to glucose tracking or insulin delivery.')}
FIX_V2 = {1: ('treatment', 'This is about updating the treatment plan to match a new prescription, not a malfunction.', 'P3', "The app's records are temporarily out of sync with the actual prescription, but the patient still has clear guidance from their endocrinologist."), 2: ('device', 'The sensor is failing to maintain a connection to the app, which is a hardware or connectivity malfunction.', 'P1', 'The user has disabled their overnight alert entirely in response to the disconnections, removing their safety net for detecting a dangerous low overnight.'), 3: ('device', 'The transmitter has stopped functioning entirely, which is a hardware failure.', 'P1', 'The patient has zero visibility into their glucose levels, which is a complete loss of monitoring.'), 4: ('treatment', 'This is a request to update the app to reflect a new dosing regimen from the provider.', 'P3', 'This is a configuration update tied to a real regimen change, not a malfunction, and the patient still has clear guidance from their provider.'), 5: ('lifestyle', 'This is a general food and activity question, not tied to a specific dose.', 'P4', 'This has no bearing on glucose tracking or insulin delivery.'), 6: ('account', 'This is a request to transfer profile and settings across devices, an administrative task.', 'P4', 'This is a routine account transfer request with no impact on glucose tracking or insulin delivery.'), 7: ('supply', 'This is about getting a replacement for a damaged accessory.', 'P3', "The current pen cap is still functioning, so there's a buffer before this becomes urgent."), 8: ('caregiver', "This is about managing who has authorized access to the account, not the account's own billing or profile.", 'P3', "Removing unwanted access is important, but the patient's own monitoring and alerts are unaffected in the meantime."), 9: ('account', 'This is a login and access issue, not a device malfunction.', 'P2', 'Being locked out means the patient currently has no way to view their glucose data, which is the same kind of monitoring gap as a connectivity loss.'), 10: ('device', 'This is a data accuracy bug in a tracking feature within the app.', 'P4', 'This is a cosmetic discrepancy unrelated to glucose tracking or insulin delivery.')}

def extract_json(txt):
    if '```' in txt:
        parts = txt.split('```')
        for part in parts:
            part = part.strip()
            if part.startswith('json'):
                part = part[4:].strip()
            if part.startswith('{'):
                return part
    start = txt.find('{')
    end = txt.rfind('}')
    if start != -1 and end != -1 and end > start:
        return txt[start:end+1]
    return txt

def run_case(version, prompt, t):
    if LIVE:
        txt = gemini_chat([{'role':'user','content': prompt + '\nTicket: ' + t['ticket']}])
        try:
            cleaned = extract_json(txt)
            d = json.loads(cleaned)
            return d.get('category',''), d.get('category_rationale',''), d.get('severity',''), d.get('severity_rationale','')
        except Exception:
            return '', txt or '', '', ''
    fix = FIX_V1 if version == 'v1' else FIX_V2
    return fix[t['id']]

def score(version, prompt):
    rows = []
    sev_key = 'expected_severity_v1' if version == 'v1' else 'expected_severity_v2'
    for t in tests:
        cat, cat_why, sev, sev_why = run_case(version, prompt, t)
        rationale_text = cat_why + ' ' + sev_why
        rows.append({
            'id': t['id'],
            'cat_exact': exact_match(t['expected_category'], cat),
            'sev_exact': exact_match(t[sev_key], sev),
            'sem': semantic_sim(t['why'], rationale_text),
            'got_cat': cat, 'got_sev': sev,
        })
    n = len(rows)
    cat_acc = sum(r['cat_exact'] for r in rows) / n
    sev_acc = sum(r['sev_exact'] for r in rows) / n
    avg_sem = sum(r['sem'] for r in rows) / n
    return cat_acc, sev_acc, avg_sem, rows

cat1, sev1, sem1, r1 = score('v1', PROMPT_V1)
cat2, sev2, sem2, r2 = score('v2', PROMPT_V2)
print(f'v1  category exact-match {cat1:.0%}   severity exact-match {sev1:.0%}   avg semantic-sim {sem1:.3f}')
print(f'v2  category exact-match {cat2:.0%}   severity exact-match {sev2:.0%}   avg semantic-sim {sem2:.3f}')

v1  category exact-match 90%   severity exact-match 50%   avg semantic-sim 0.430
v2  category exact-match 90%   severity exact-match 80%   avg semantic-sim 0.411


## Part 3 and 4: the tradeoff and the failure
Live run against Gemini surfaced a different and more useful outcome than the fixture predictions did. Category held steady on 9 of 10 cases; the one drift (#3) is the required failure. Severity accuracy improved substantially between versions, but not on the two cases originally predicted.  Instead, v1 already reasoned correctly about #2 and #9 on its own, and v2's real and unplanned escalation occurred on #7.

In [5]:
print('Category stability (v1 -> v2):')
for t in tests:
    c1 = next(r for r in r1 if r['id']==t['id'])['got_cat']
    c2 = next(r for r in r2 if r['id']==t['id'])['got_cat']
    if c1 != c2:
        print(f"  #{t['id']} DRIFTED: v1={c1} -> v2={c2}")
print('  (no output above means category held steady on every case)')

print()
print('Severity (v1 -> v2):')
for t in tests:
    s1 = next(r for r in r1 if r['id']==t['id'])['got_sev']
    s2 = next(r for r in r2 if r['id']==t['id'])['got_sev']
    marker = '  <-- escalation' if s1 != s2 else ''
    print(f"  #{t['id']}: v1={s1}  v2={s2}{marker}")
# TODO (you): explain why v2 helped one case and hurt another, and how you would resolve the tradeoff.

Category stability (v1 -> v2):
  (no output above means category held steady on every case)

Severity (v1 -> v2):
  #1: v1=P3  v2=P3
  #2: v1=P1  v2=P1
  #3: v1=P2  v2=P2
  #4: v1=P3  v2=P3
  #5: v1=P4  v2=P4
  #6: v1=P4  v2=P4
  #7: v1=P4  v2=P3  <-- escalation
  #8: v1=P4  v2=P4
  #9: v1=P2  v2=P2
  #10: v1=P4  v2=P4


### Metric results

Run 1: v1 category 90%, severity 50%, sem-sim 0.443. v2 category 80%, severity 80%, sem-sim 0.468.

Run 2: v1 category 90%, severity 50%, sem-sim 0.430. v2 category 90%, severity 80%, sem-sim 0.411.

### The tradeoff

Severity accuracy jumped from 50% to 80%, which is what was what I was looking for with the addition of the downstream rule. Category accuracy dropped from 90% to 80%, and that drop was ticket #3, but held at 90% in run 2 without changing the category. That's a quantified tradeoff where v2 buys a severity improvement, but whether it also causes a category miss depends on the run, in run 1 it dropped one category call, whilst in run 2 it didn't.

### Case #3: the required failure

The CGM transmitter died completely. Run 1, v1 called it device and v2 called it supply. Run 2, both called it device. The category switch didn't hold up so that was probably noise. The severity call did hold up, and it was wrong both times. v1 scored this P2 instead of P1, in both runs, and v2 made the same mistake both times. v1's own table lists complete loss of critical alerts under P1, and total transmitter death fits that description. My downstream rule didn't catch it either. If I were to carry on this experiment, there would need to be a v3 prompt version that expanded on categorizing patient risk with rationale.

### Things I didn't expect or predict

I assumed v1 would miss the danger in the sleep-disruption ticket (#2) and the account lockout ticket (#9), scoring them too low, and that v2's downstream rule would catch them. Live, v1 already scored both correctly, P1 and P2. The model reasoned about the consequence on its own even without my instructions telling it to. That demonstrates that a capable model already does some of this thinking by default, and my literal only v1 prompt didn't stop it from happening the way I had predicted.

### Case #7: Unexpected escalation

The cracked pen cap ticket went from P4 under v1 to P3 under v2, and I didn't build this case to test that. Finding a real, unplanned severity bump is better evidence that the downstream rule generalizes vs either of my hand-picked cases.

### Summary

My fixture-based predictions were built around a hypothesis that turned out to be partly wrong. If I'd submitted on fixtures, I would have written a research note about two escalation cases that live aren't escalations at all, confirmed across two separate runs. I also would have missed that both versions get #3's severity wrong the same way, since my fixtures assumed even v1 would obviously score total transmitter failure as P1. Real model output surfaced a shared blind spot I hadn't thought to predict, and disproved part of what I assumed going in.